In [1]:
import sys
sys.path.insert(0, '../../')

In [2]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
from sklearn.base import clone
from tqdm import tqdm
from pickle import dump

# load local variables
from src.config import *
from src.load_models import select_model
from src.utils import calculate_r2_score, calculate_per_diff, calculate_combined_r2_and_per_diff, find_adj_score, combine_all_batches, split_batches_back, perform_combat_normalization
from src.utils import calculate_y_LOD, find_score, calculate_r2_score_KFold, calculate_per_diff_KFold
from src.feature_selection import ModelSelection
from src.graph_visualization import feature_selection_tabularize, create_correlation_matrix
from src.outlier_detection import find_outliers_remove

from scipy.stats import ttest_ind

In [39]:
def perform_ttest(grp1:pd.DataFrame, 
                  grp2:pd.DataFrame, 
                  feature_list:list):
    df = pd.DataFrame(columns=['Features', 't-statistics', 'p-value'])
    for feature in feature_list:
        t_stat, p_value = ttest_ind(grp1[feature].tolist(), grp2[feature].tolist())
        df = pd.concat([df, pd.DataFrame({'Features':[feature], 't-statistics':[t_stat], 'p-value':[round(p_value, 5)]})], ignore_index=True)
        
    return df

In [40]:
# Load Extracted features dataset
dataset_root = '../../dataset'
ML6_1 = pd.read_excel(f'{dataset_root}/ML6/Day 1 Data/feature_extraction_noise_None.xlsx')
ML6_2 = pd.read_excel(f'{dataset_root}/ML6/Day 2 Data/feature_extraction_noise_None.xlsx')

ML5   = pd.read_excel(f'{dataset_root}/ML5/feature_extraction_noise_None.xlsx')

ML1   = pd.read_excel(f'{dataset_root}/2024_02_19_ML1/feature_extraction_noise_None.xlsx')
ML2   = pd.read_excel(f'{dataset_root}/2024_02_22_ML2/feature_extraction_noise_None.xlsx')
    
ML6_1['dataset_name'] = 'ML6-1'
ML6_2['dataset_name'] = 'ML6-2'
ML1['dataset_name']   = 'ML1'
ML2['dataset_name']   = 'ML2'
ML5['dataset_name']   = 'ML5'

In [41]:
final_dataset = pd.concat([ML6_1, ML6_2, ML5, ML1, ML2], ignore_index=True)
final_dataset['label'] = final_dataset['file'].apply(lambda x: x.split('/')[-1].split('_')[-2].replace('cbz','')).apply(lambda x: int(x))
final_dataset.rename(columns={"PH": 'max(S)', 'signal_std':'std(S)', 'signal_mean':'mean(S)', 'peak area':'area(S)', \
                        'dS_dV_area':'area(dS/dV)', 'dS_dV_max_peak':'max(dS/dV)', 'dS_dV_min_peak':'min(dS/dV)',\
                    'dS_dV_peak_diff':'max(dS/dV) - min(dS/dV)', \
                    'peak V':'V_max(S)', 'dS_dV_max_V':'V_max(dS/dV)', 'dS_dV_min_V':'V_min(dS/dV)',\
        }, inplace = True)

final_dataset = final_dataset[['area(S)', 'max(S)', 'max(dS/dV)', 'min(dS/dV)', 'V_max(S)', 'vcenter', 'V_max(dS/dV)', 'V_min(dS/dV)', 'f1', 'f2', 'file', 'label', 'dataset_name']]

In [44]:
# Perform t-test between ML6 day-1 and ML6 day-2
feature_ttest = ['area(S)', 'max(S)', 'max(dS/dV)', 'min(dS/dV)', 'V_max(S)', 'vcenter', 'V_max(dS/dV)', 'V_min(dS/dV)', 'f1', 'f2']
grp1          = final_dataset[(final_dataset['dataset_name']=='ML6-1') | (final_dataset['dataset_name']=='ML6-2')]
grp2          = final_dataset[(final_dataset['dataset_name']=='ML5') | (final_dataset['dataset_name']=='ML2')]

In [45]:
perform_ttest(grp1, grp2, feature_ttest)

,Features,t-statistics,p-value
0,area(S),-1.322522,0.18666
1,max(S),-0.887411,0.37532
2,max(dS/dV),-1.327582,0.18498
3,min(dS/dV),0.010552,0.99159
4,V_max(S),1.320954,0.18718
5,vcenter,0.659435,0.50995
6,V_max(dS/dV),0.515589,0.60639
7,V_min(dS/dV),1.444524,0.14928
8,f1,-0.307240,0.75880
9,f2,-5.442582,0.00000
